# 33. Neural Networks: Perceptron and Multilayer Perceptron (MLP)

## Algorithm Category
**Type**: Neural Networks - Feedforward  
**Complexity**: Medium  
**Use Case**: Basic neural network architecture for classification and regression

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the perceptron and its limitations
- Implement a single-layer perceptron from scratch
- Understand multilayer perceptrons (MLPs) and backpropagation
- Build MLPs using scikit-learn and PyTorch
- Visualize decision boundaries and network architecture
- Apply MLPs to classification and regression problems

## Historical Context

Perceptron was developed by Rosenblatt in 1957:
- Rosenblatt, F. (1957): "The Perceptron: A Perceiving and Recognizing Automaton"
- First artificial neuron model
- Foundation for modern neural networks

**Key Papers/References:**
- Rosenblatt, F. (1957). "The Perceptron: A Perceiving and Recognizing Automaton"
- Rumelhart, D.E., et al. (1986). "Learning representations by back-propagating errors"

## When to Use Perceptron and MLP

Perceptron/MLP is appropriate when:
- You have tabular/structured data
- Non-linear relationships in data
- Classification or regression tasks
- Moderate-sized datasets
- Need interpretable neural network
- Good starting point for deep learning

## Theory & Mechanics

### Mathematical Foundation

**Single Perceptron:**
$$y = f(\sum_{i=1}^{n} w_i x_i + b)$$

Where:
- $w_i$: Weights
- $x_i$: Input features
- $b$: Bias
- $f$: Activation function (step, sigmoid, ReLU, etc.)

**Multilayer Perceptron (MLP):**
- Input layer: Receives features
- Hidden layers: Process information
- Output layer: Produces predictions

**Forward Propagation:**
$$h^{(l)} = f(W^{(l)} h^{(l-1)} + b^{(l)})$$

**Backpropagation:**
- Compute gradients using chain rule
- Update weights using gradient descent
- Propagate errors backward through network

### How It Works

1. **Initialize**: Random weights and biases
2. **Forward pass**: Compute activations layer by layer
3. **Compute loss**: Compare predictions with targets
4. **Backward pass**: Calculate gradients
5. **Update weights**: Gradient descent step
6. **Repeat**: Steps 2-5 until convergence

### Key Hyperparameters

- **hidden_layer_sizes**: Number of neurons in each hidden layer
- **activation**: Activation function ('relu', 'tanh', 'logistic')
- **solver**: Optimization algorithm ('adam', 'sgd', 'lbfgs')
- **learning_rate**: Step size for weight updates
- **max_iter**: Maximum iterations
- **alpha**: L2 regularization parameter

### Advantages

- Can learn non-linear patterns
- Universal function approximator (with enough neurons)
- Works with various data types
- Interpretable architecture
- Foundation for deep learning

### Limitations

- Requires careful hyperparameter tuning
- Can overfit easily
- Slow for very large datasets
- May get stuck in local minima
- Requires feature scaling


## Implementation

Let's implement perceptron and MLP.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_circles, load_iris
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error

# Import our helper functions
from src.models.supervised import split_data, evaluate_classifier, evaluate_regressor
from src.models.classification import calculate_classification_metrics

print("Libraries imported successfully!")


In [ ]:
# Simple Perceptron Implementation
class SimplePerceptron:
    def __init__(self, learning_rate=0.1, max_iter=1000):
        self.lr = learning_rate
        self.max_iter = max_iter
    
    def fit(self, X, y):
        """Train perceptron"""
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Convert labels to -1 and 1
        y_binary = np.where(y == 0, -1, 1)
        
        for _ in range(self.max_iter):
            errors = 0
            for i in range(n_samples):
                # Prediction
                output = np.dot(X[i], self.weights) + self.bias
                prediction = 1 if output >= 0 else -1
                
                # Update if misclassified
                if prediction != y_binary[i]:
                    self.weights += self.lr * y_binary[i] * X[i]
                    self.bias += self.lr * y_binary[i]
                    errors += 1
            
            if errors == 0:
                break
    
    def predict(self, X):
        """Make predictions"""
        output = np.dot(X, self.weights) + self.bias
        return np.where(output >= 0, 1, 0)

# Test on simple dataset
X_simple, y_simple = make_classification(n_samples=100, n_features=2, n_redundant=0, 
                                         n_informative=2, n_clusters_per_class=1, 
                                         random_state=42)
X_simple = StandardScaler().fit_transform(X_simple)

perceptron = SimplePerceptron(learning_rate=0.1, max_iter=1000)
perceptron.fit(X_simple, y_simple)
y_pred_simple = perceptron.predict(X_simple)
accuracy = accuracy_score(y_simple, y_pred_simple)

print(f"Simple Perceptron Results:")
print(f"  Accuracy: {accuracy:.3f}")
print(f"  Weights: {perceptron.weights}")
print(f"  Bias: {perceptron.bias:.3f}")


In [ ]:
# Visualize perceptron decision boundary
plt.figure(figsize=(10, 6))
plt.scatter(X_simple[y_simple == 0, 0], X_simple[y_simple == 0, 1], 
           c='red', marker='o', label='Class 0', alpha=0.7)
plt.scatter(X_simple[y_simple == 1, 0], X_simple[y_simple == 1, 1], 
           c='blue', marker='s', label='Class 1', alpha=0.7)

# Plot decision boundary
x_min, x_max = X_simple[:, 0].min() - 0.5, X_simple[:, 0].max() + 0.5
y_min, y_max = X_simple[:, 1].min() - 0.5, X_simple[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))
Z = perceptron.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Perceptron Decision Boundary')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Multilayer Perceptron (MLP)

Let's implement MLP using scikit-learn.


In [ ]:
# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

print(f"Dataset Shape: {X.shape}")
print(f"Classes: {iris.target_names.tolist()}")

# Scale features (important for neural networks)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = split_data(X_scaled, y, test_size=0.2, random_state=42)

# Train MLP Classifier
mlp = MLPClassifier(hidden_layer_sizes=(10, 5), activation='relu', 
                   solver='adam', learning_rate_init=0.001,
                   max_iter=500, random_state=42)
mlp.fit(X_train, y_train)

print(f"\nMLP Classifier:")
print(f"  Architecture: {X.shape[1]} -> {mlp.hidden_layer_sizes} -> {len(iris.target_names)}")
print(f"  Activation: {mlp.activation}")
print(f"  Solver: {mlp.solver}")
print(f"  Number of iterations: {mlp.n_iter_}")

# Make predictions
y_pred = mlp.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy:.3f}")

# Evaluate
results = evaluate_classifier(mlp, X_test, y_test)
metrics = calculate_classification_metrics(y_test.values, y_pred)
print(f"Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1_score']:.3f}")


## Learning Curve

Let's visualize the loss during training.


In [ ]:
# Plot loss curve
if hasattr(mlp, 'loss_curve_'):
    plt.figure(figsize=(10, 6))
    plt.plot(mlp.loss_curve_)
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.title('MLP Training Loss Curve')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Final loss: {mlp.loss_curve_[-1]:.4f}")
else:
    print("Loss curve not available (may need to set warm_start=False or use different solver)")


## Comparing Different Architectures

Let's test different MLP architectures.


In [ ]:
# Test different architectures
architectures = [
    (5,),      # Single hidden layer, 5 neurons
    (10,),     # Single hidden layer, 10 neurons
    (10, 5),   # Two hidden layers: 10, 5
    (20, 10),  # Two hidden layers: 20, 10
]

results = []

for arch in architectures:
    mlp_test = MLPClassifier(hidden_layer_sizes=arch, activation='relu',
                            solver='adam', learning_rate_init=0.001,
                            max_iter=500, random_state=42)
    mlp_test.fit(X_train, y_train)
    y_pred_test = mlp_test.predict(X_test)
    acc = accuracy_score(y_test, y_pred_test)
    results.append({
        'architecture': str(arch),
        'accuracy': acc,
        'n_layers': len(arch),
        'total_neurons': sum(arch)
    })
    print(f"Architecture {arch}: Accuracy = {acc:.3f}")

# Visualize
results_df = pd.DataFrame(results)
plt.figure(figsize=(10, 6))
plt.bar(range(len(results_df)), results_df['accuracy'], alpha=0.7)
plt.xlabel('Architecture')
plt.ylabel('Accuracy')
plt.title('MLP Performance by Architecture')
plt.xticks(range(len(results_df)), results_df['architecture'], rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


## Non-Linear Classification

Let's test MLP on non-linearly separable data.


In [ ]:
# Create non-linearly separable dataset (circles)
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)
X_circles_scaled = StandardScaler().fit_transform(X_circles)

X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_circles_scaled, y_circles, test_size=0.2, random_state=42
)

# Train MLP
mlp_circles = MLPClassifier(hidden_layer_sizes=(10, 5), activation='relu',
                           solver='adam', learning_rate_init=0.001,
                           max_iter=500, random_state=42)
mlp_circles.fit(X_c_train, y_c_train)

y_c_pred = mlp_circles.predict(X_c_test)
acc_circles = accuracy_score(y_c_test, y_c_pred)

print(f"Circles Dataset:")
print(f"  Test Accuracy: {acc_circles:.3f}")

# Visualize decision boundary
plt.figure(figsize=(10, 6))
h = 0.02
x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

Z = mlp_circles.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
plt.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', 
           edgecolors='black', s=50)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('MLP Decision Boundary (Non-Linear)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Validation & Testing

Let's validate the model and compare activation functions.


In [ ]:
# Compare activation functions
activations = ['identity', 'logistic', 'tanh', 'relu']
activation_results = []

for act in activations:
    mlp_act = MLPClassifier(hidden_layer_sizes=(10, 5), activation=act,
                           solver='adam', learning_rate_init=0.001,
                           max_iter=500, random_state=42)
    mlp_act.fit(X_train, y_train)
    y_pred_act = mlp_act.predict(X_test)
    acc = accuracy_score(y_test, y_pred_act)
    activation_results.append({'activation': act, 'accuracy': acc})
    print(f"Activation {act}: Accuracy = {acc:.3f}")

# Assertions
assert accuracy > 0.5, "MLP should perform better than random"
assert acc_circles > 0.5, "MLP should handle non-linear data"
print("\n✓ Validation checks passed")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Perceptron Basics**
   - Single-layer neural network
   - Linear classifier
   - Can only learn linearly separable patterns
   - Foundation for neural networks

2. **Multilayer Perceptron (MLP)**
   - Multiple layers of neurons
   - Can learn non-linear patterns
   - Universal function approximator
   - Uses backpropagation for training

3. **Key Components**
   - **Weights**: Connections between neurons
   - **Biases**: Offset terms
   - **Activation functions**: Introduce non-linearity (ReLU, sigmoid, tanh)
   - **Loss function**: Measures prediction error
   - **Optimizer**: Updates weights (SGD, Adam, etc.)

4. **Best Practices**
   - Always scale features before training
   - Start with simple architectures
   - Use ReLU activation for hidden layers
   - Monitor loss curve for convergence
   - Use regularization to prevent overfitting

### When to Use Perceptron and MLP

✅ **Good for:**
- Tabular/structured data
- Classification and regression
- Non-linear relationships
- Moderate-sized datasets
- When you need neural network interpretability
- Starting point for deep learning

❌ **Not ideal for:**
- Image data (use CNNs)
- Sequential data (use RNNs)
- Very large datasets (use deep learning)
- When linear models suffice
- Real-time applications (can be slow)

### Next Steps

- Explore **Deep Neural Networks** with more layers
- Try **Convolutional Neural Networks** for images
- Use **Recurrent Neural Networks** for sequences
- Apply **Regularization** techniques (dropout, L2)
- Experiment with **different optimizers** (Adam, RMSprop)
